In [ ]:
from pathlib import Path
import sys

REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'ess').is_dir() and (p / 'data/cstr').is_dir()), None)  # Repository root, independent of the notebook's working directory.
if REPO_ROOT is None:
    raise FileNotFoundError('Open this notebook from inside the repository.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from ess import ESSConfig, LearningRateConfig, FineTuningConfig, run_experiment

In [ ]:
config = ESSConfig(  # Settings for one fresh CSTR training run.
    data_dir='data/cstr',  # Directory containing the CSTR MAT files.
    output_dir='outputs/training',  # Parent directory for separate experiment outputs.
    training_file='training_reference.mat',  # Historical training reference, error and PID actions.
    validation_file='validation_reference.mat',  # Historical validation reference, error and PID actions.
    training_excitation='training_disturbances.mat',  # Training flow and concentration disturbances.
    validation_excitation='validation_disturbances.mat',  # Validation flow and concentration disturbances.
    pid_file='pid_parameters.mat',  # MAT file containing the teacher PID parameters.
    sample_time_min=0.01,  # Sampling interval in minutes.
    dataset_fraction=1.0,  # Leading fraction of each trajectory; 1.0 uses all samples.
    architecture='LeakyMLP4',  # Controller architecture: LeakyMLP4, MLP8 or LSTM.
    lstm_window=500,  # Number of normalized feature vectors in each LSTM input window.
    error_window=300,  # Number of errors in the sliding-sum feature.
    error_retention=1.0,  # Memory retention for accumulated error; 1.0 gives a running sum.
    epochs=308,  # Number of scheduled-sampling training epochs.
    accumulation_steps=128,  # Samples averaged before each optimizer update.
    seed=0,  # Seed shared by Python, NumPy and PyTorch.
    deterministic=True,  # Require deterministic PyTorch algorithms.
    device='auto',  # Execution device: auto, cpu or cuda.
    sampling_schedule='inverse_sigmoid',  # ANN probability schedule: inverse_sigmoid or linear.
    weighted_actions=True,  # Blend PID and ANN actions when the ANN branch is sampled.
    action_noise_std=0.1,  # Gaussian noise standard deviation in normalized ANN-action units.
    gradient_clip_norm=3.0,  # Maximum averaged gradient norm; None disables clipping.
    weight_decay=0.001,  # Adam weight-decay coefficient; 0 disables it.
    sequence_score_enabled=True,  # Include the detached recent-action error score in reported loss.
    sequence_score_weight=0.1,  # Weight of the detached sequence score.
    sequence_length=15,  # Previous action pairs included in the sequence score.
    middle_autonomy=0.3,  # ANN probability at which the middle LR cap begins.
    late_autonomy=0.95,  # ANN probability at which scheduling switches to validation loss.
    selection_min_autonomy=0.95,  # Minimum ANN probability for final-checkpoint eligibility.
    validation_target='historical_pid',  # Autonomous-rollout comparison target: historical_pid or live_pid.
    plot_every=20,  # Plot trajectories and losses at this interval, plus first/last epochs; 0 disables epoch plots.
    checkpoint_every=20,  # Epoch interval for extra weight snapshots; 0 disables them.
    show_plots=True,  # Display saved figures in the notebook.
    export_validation=True,  # Save the selected model’s autonomous trajectory to CSV.
    lr=LearningRateConfig(  # Stage and plateau learning-rate settings.
        early=0.001,  # Early-phase learning-rate cap.
        middle=0.0001,  # Middle-phase learning-rate cap.
        late=2e-05,  # Late-phase learning-rate cap.
        minimum=2e-06,  # Global learning-rate lower bound.
        maximum=0.001,  # Global learning-rate upper bound.
        plateau_patience=50,  # Non-improving epochs tolerated before a reduction.
        plateau_factor=0.5,  # Multiplier applied at each plateau reduction.
    ),
    fine_tuning=FineTuningConfig(  # Optional final high-autonomy training settings.
        enabled=False,  # Enable a separate stage starting from the eligible best ESS model.
        epochs=50,  # Number of fine-tuning epochs.
        autonomy=0.99,  # Fixed ANN probability and blending weight, between 0.99 and 1.0.
        learning_rate=1e-05,  # Initial fine-tuning learning rate.
        action_noise_std=0.01,  # Normalized action noise; 0 disables noise.
    ),
)

In [ ]:
result = run_experiment(config, repo_root=REPO_ROOT)  # Outputs and checkpoint-selection metadata for this run.
print(result.directory)